# Cyclone Risk Assessment Training Pipeline (North Indian Ocean)
This Jupyter notebook trains the machine learning model for **North Indian Ocean Cyclone Risk Assessment** using historical IBTrACS storm tracks or a synthetic generator fallback.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

from hazard_cyclone.train import train_and_select_model
from hazard_cyclone.assess import assess_risk

### Step 1: Train Model & Evaluate Candidates
We use GroupShuffleSplit grouped by storm ID (`SID`) to prevent temporal track leakage.

In [2]:
artifact = train_and_select_model()
print('Training Complete!')
print('Model Name:', artifact['model_name'])
print('Data Source:', artifact['training_data_source'])
print('Validation Metrics:', artifact['validation_metrics'])

Training Complete!
Model Name: logistic_regression
Data Source: real
Validation Metrics: {'calibrated_hist_gbdt': {'pr_auc': 0.7278, 'roc_auc': 0.9378, 'brier_score': 0.052, 'positive_rate': 0.1062}, 'logistic_regression': {'pr_auc': 0.7357, 'roc_auc': 0.9388, 'brier_score': 0.0511, 'positive_rate': 0.1062}}


### Step 2: Test Live Risk Assessment
Assess risk for a target location (e.g., Puri, Odisha) against an active storm feed.

In [3]:
result = assess_risk(
    location={'lat': 19.81, 'lon': 85.83, 'name': 'Puri, Odisha'},
    observation_time='2026-11-16T18:00:00Z'
)
import json
print(json.dumps(result, indent=2))

{
  "hazard_type": "cyclone",
  "location": {
    "lat": 19.81,
    "lon": 85.83,
    "name": "Puri, Odisha"
  },
  "assessed_at": "2026-11-16T19:46:24Z",
  "valid_for": "2026-11-16T18:00:00Z",
  "forecast_horizon_hours": 48,
  "risk_score": 88.4,
  "risk_level": "Critical",
  "confidence": 0.82,
  "key_indicators": {
    "active_storms_count": 1,
    "dominant_storm_id": "BOB_2026_01",
    "dominant_storm_name": "DANA",
    "nearest_storm_distance_km": 206.0,
    "max_wind_kt": 72.0,
    "probability_50kt_48h": 0.9822,
    "distance_to_coast_km": 0.6,
    "sub_basin": "Bay of Bengal"
  },
  "model_info": {
    "model_name": "logistic_regression",
    "model_version": "1.0.0",
    "trained_on": "NIO Cyclone Tracks (real)"
  },
  "data_source": "simulated",
  "limitations": [
    "Evaluates 48h wind impact probability for existing active NIO storms."
  ]
}
